# ZKAuth: Reproduce Figures 4, 5, and 6

Loads the result files produced by `src/evaluation/run_simulation.py`
(fixed `seed=42`) and reproduces the three results figures of the paper:
PPAS across rounds (Fig. 4), attack resistance (Fig. 5), and
scalability (Fig. 6). All error bars are 95% confidence intervals.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

RESULTS = Path('..') / 'data' / 'results'
raw = pd.read_csv(RESULTS / 'raw_rounds.csv')
agg = pd.read_csv(RESULTS / 'aggregate_ppas.csv')
att = pd.read_csv(RESULTS / 'attack_resistance.csv')
sca = pd.read_csv(RESULTS / 'scalability.csv')

SCHEMES = ['ZKAuth (Proposed)', 'ZK-SNARK (Groth16)', 'ZK-STARK', 'Bulletproofs']
MARKERS = {'ZKAuth (Proposed)': 'o', 'ZK-SNARK (Groth16)': 's',
           'ZK-STARK': '^', 'Bulletproofs': 'D'}
t_crit = stats.t.ppf(0.975, df=999)  # per-round n_auth = 1000
agg[['scheme', 'ppas_grand_mean', 'ppas_ci95']]

## Figure 4: PPAS across evaluation rounds (95% CI)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for s in SCHEMES:
    d = raw[raw.scheme == s].sort_values('round')
    ci = t_crit * d.ppas_std / np.sqrt(d.n_auth)
    ax.errorbar(d['round'], d.ppas_mean, yerr=ci, marker=MARKERS[s],
                capsize=2, label=s)
ax.set_xlabel('Evaluation Round'); ax.set_ylabel('PPAS')
ax.set_xticks(range(1, 11)); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Figure 5: Adversarial success rate per attack vector (95% CI)

In [ ]:
attacks = ['Replay Attack', 'Correlation Attack', 'Linkability Attack', 'Sybil Attack']
x = np.arange(len(attacks)); w = 0.2
fig, ax = plt.subplots(figsize=(8, 4))
for i, s in enumerate(SCHEMES):
    d = att[att.scheme == s].set_index('attack').loc[attacks]
    ax.bar(x + (i - 1.5) * w, 100 * d.success_rate_mean, w,
           yerr=100 * d.success_rate_ci95, capsize=2, label=s)
ax.set_xticks(x); ax.set_xticklabels(attacks, rotation=10)
ax.set_ylabel('Adversarial Success Rate (%)'); ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## Figure 6: Scalability (verification latency and PPAS vs registry size)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for s in SCHEMES:
    d = sca[sca.scheme == s].sort_values('n_identities')
    a1.errorbar(d.n_identities, d.verify_ms, yerr=d.verify_ms_ci95,
                marker=MARKERS[s], capsize=2, label=s)
    a2.errorbar(d.n_identities, d.ppas, yerr=d.ppas_ci95,
                marker=MARKERS[s], capsize=2, label=s)
for a, yl in ((a1, 'Verification Latency (ms)'), (a2, 'PPAS')):
    a.set_xscale('log'); a.set_xlabel('DID Registry Size')
    a.set_ylabel(yl); a.grid(alpha=0.3)
a1.legend(fontsize=8); plt.tight_layout(); plt.show()